In [1]:
import json
import random
from pathlib import Path
from collections import Counter

In [4]:

OUTPUT_DIR = Path("../data2")
FINAL_INTERACTION_FILE = OUTPUT_DIR / "final_interactions.jsonl"

HF_RELEASE_DIR = OUTPUT_DIR / "hf_release"

INTERACTIONS_DIR = HF_RELEASE_DIR / "interactions"
METADATA_DIR = HF_RELEASE_DIR / "metadata"

INTERACTIONS_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = INTERACTIONS_DIR / "train.jsonl"
VALIDATION_FILE = INTERACTIONS_DIR / "validation.jsonl"
TEST_FILE = INTERACTIONS_DIR / "test.jsonl"

FINAL_METADATA_FILE = OUTPUT_DIR / "final_metadata.jsonl"
HF_METADATA_FILE = METADATA_DIR / "final_metadata.jsonl"

In [5]:
EXPECTED_TOTAL = 72_909

TRAIN_SIZE = 58_327
VALIDATION_SIZE = 7_291
TEST_SIZE = 7_291

SEED = 42

In [6]:

# =========================================================
# Load final interaction records
# =========================================================

records = []

with open(FINAL_INTERACTION_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        records.append(json.loads(line))

In [7]:
# =========================================================
#  Validate total count
# =========================================================

print("Loaded records:", len(records))

assert len(records) == EXPECTED_TOTAL, (
    f"Expected {EXPECTED_TOTAL} records, "
    f"but found {len(records)}"
)

Loaded records: 72909


In [8]:
# =========================================================
# Check user IDs
# =========================================================

user_ids = [record["user_id"] for record in records]

unique_users = set(user_ids)

print("Unique users:", len(unique_users))

assert len(unique_users) == EXPECTED_TOTAL, (
    "Duplicate user_id found. "
    "Do not split until this is resolved."
)

Unique users: 72909


In [9]:

random.seed(SEED)
random.shuffle(records)

In [10]:
# =========================================================
# Exact split
# =========================================================

train_records = records[:TRAIN_SIZE]

validation_records = records[
    TRAIN_SIZE:
    TRAIN_SIZE + VALIDATION_SIZE
]

test_records = records[
    TRAIN_SIZE + VALIDATION_SIZE:
]

In [11]:
assert len(train_records) == TRAIN_SIZE
assert len(validation_records) == VALIDATION_SIZE
assert len(test_records) == TEST_SIZE

assert (
    len(train_records)
    + len(validation_records)
    + len(test_records)
    == EXPECTED_TOTAL
)

In [12]:

def write_jsonl(path, data):
    with open(path, "w", encoding="utf-8") as f:
        for record in data:
            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                ) + "\n"
            )

In [13]:
write_jsonl(TRAIN_FILE, train_records)
write_jsonl(VALIDATION_FILE, validation_records)
write_jsonl(TEST_FILE, test_records)

In [14]:
import shutil

shutil.copy2(
    FINAL_METADATA_FILE,
    HF_METADATA_FILE
)

WindowsPath('../data2/hf_release/metadata/final_metadata.jsonl')

In [15]:
print("\n==============================")
print("FINAL DATASET SPLIT")
print("==============================")

print("Train      :", len(train_records))
print("Validation :", len(validation_records))
print("Test       :", len(test_records))
print("Total      :", (
    len(train_records)
    + len(validation_records)
    + len(test_records)
))

print("\nRelease directory:")
print(HF_RELEASE_DIR)


FINAL DATASET SPLIT
Train      : 58327
Validation : 7291
Test       : 7291
Total      : 72909

Release directory:
..\data2\hf_release


In [17]:
! pip install -q -U huggingface_hub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 5.0.1 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.1.1 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
from huggingface_hub import login

login()

In [19]:
HF_DATASET_ID = "0xMeet/LLM-Personalized-Recommendation-Dataset"

print(HF_DATASET_ID)

0xMeet/LLM-Personalized-Recommendation-Dataset


In [20]:
from huggingface_hub import HfApi

api = HfApi()

api.create_repo(
    repo_id=HF_DATASET_ID,
    repo_type="dataset",
    private=False,
    exist_ok=True
)

print("Dataset repository created ✅")
print(f"https://huggingface.co/datasets/{HF_DATASET_ID}")

Dataset repository created ✅
https://huggingface.co/datasets/0xMeet/LLM-Personalized-Recommendation-Dataset


In [22]:
from huggingface_hub import HfApi

api = HfApi()

info = api.repo_info(
    repo_id=HF_DATASET_ID,
    repo_type="dataset"
)

print("Repository found ✅")
print(info.id)

Repository found ✅
0xMeet/LLM-Personalized-Recommendation-Dataset


In [23]:
README_FILE = HF_RELEASE_DIR / "README.md"

print(README_FILE)

..\data2\hf_release\README.md


In [24]:
for path in HF_RELEASE_DIR.rglob("*"):
    if path.is_file():
        print(path.relative_to(HF_RELEASE_DIR))

README.md
interactions\test.jsonl
interactions\train.jsonl
interactions\validation.jsonl
metadata\final_metadata.jsonl


In [25]:
api.upload_folder(
    repo_id=HF_DATASET_ID,
    repo_type="dataset",
    folder_path=str(HF_RELEASE_DIR)
)

print("Upload completed ✅")
print(
    f"https://huggingface.co/datasets/{HF_DATASET_ID}"
)

Upload completed ✅
https://huggingface.co/datasets/0xMeet/LLM-Personalized-Recommendation-Dataset
